In [3]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [19]:
from adapters.extractors.oai_pmh.axiell.runtime import AXIELL_CONFIG as ADAPTER_CONFIG
from adapters.extractors.oai_pmh.axiell import config
from lxml import etree

ROOT = "/Users/brychtas/Documents/all-raw-axiell-works-exhibition"

AXIELL_CLIENT = ADAPTER_CONFIG.build_oai_client()

In [21]:
FORMAT = "oai_raw"

for i, record in enumerate(AXIELL_CLIENT.list_records(metadata_prefix=FORMAT, set_spec="collect&gt;museum")):
    identifier = record.header.identifier.replace(":", "_")
    print(identifier)
    continue
    
    try:
        formatted_record = etree.tostring(record.metadata, pretty_print=True, encoding="unicode")
        with open(f"{ROOT}/{identifier}.xml", "w") as f:
            f.write(formatted_record)
    except Exception as e:
        print(f"Failed to parse record {record}")
            
    if i % 5000 == 0:
        print(i)

BadArgumentError: Set 'collect&gt;museum' does not exist.

In [2]:
import os
from pathlib import Path
import json

calm_works_folder = "/Users/brychtas/Documents/all-raw-calm-works"
calm_file_names = [f.name for f in Path(calm_works_folder).iterdir() if f.is_file()]

calm_works = []
for file_name in calm_file_names:
    with open(os.path.join(calm_works_folder, file_name)) as f:
        contents = json.load(f)
        calm_works.append(contents)

In [16]:
import sys
import os
import logging
from collections import Counter
from datetime import datetime, timedelta, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import chain

from oai_pmh_client.exceptions import NoRecordsMatchError



def get_oai_raw(record_id: str):
    record = AXIELL_CLIENT.get_record(identifier=record_id, metadata_prefix=FORMAT)
    return etree.tostring(record.metadata, pretty_print=True, encoding="unicode")
 

def _fetch_window(from_dt, until_dt):
    try:
        records = list(AXIELL_CLIENT.list_records(
            from_date=from_dt, until_date=until_dt,
            metadata_prefix=FORMAT, set_spec="collect"
        ))
        print(f"Fetched {len(records)} records")
        return records
    except NoRecordsMatchError as e:
        return []
    except Exception as e:
        print(e)
        return []



def build_windows():
    from_date = datetime.strptime("2026-04-20 14:30:00", "%Y-%m-%d %H:%M:%S")
    until_date = datetime.strptime("2026-06-15 14:30:00", "%Y-%m-%d %H:%M:%S")
    window_size = timedelta(hours=2)

    # Build list of (start, end) windows
    windows = []
    current = from_date
    while current < until_date:
        end = min(current + window_size, until_date)
        windows.append((current, end))
        current = end

    return windows


def fetch_records_in_parallel(max_workers: int):
    all_records = []
    windows = build_windows()
    print(f"Created {len(windows)} windows")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_fetch_window, s, e): (s, e) for s, e in windows}
        for future in as_completed(futures):
            all_records.extend(future.result())

    return all_records

In [14]:
import boto3
import time

athena = boto3.client('athena')

def parse_athena_results(response):
    rows = response["ResultSet"]["Rows"]

    # First row = column headers
    headers = [col.get("VarCharValue") for col in rows[0]["Data"]]

    data = []
    for row in rows[1:]:
        values = [col.get("VarCharValue") for col in row["Data"]]
        data.append(dict(zip(headers, values)))

    return data

def run_axiell_adapter_query(query: str):
    response = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={
            "Catalog": "s3tablescatalog/wellcomecollection-platform-axiell-adapter",
            'Database': "wellcomecollection_catalogue"
        }
    )
    query_execution_id = response['QueryExecutionId']
    
    while True:
        status = athena.get_query_execution(QueryExecutionId=query_execution_id)
        state = status['QueryExecution']['Status']['State']
    
        if state in ['SUCCEEDED', 'FAILED', 'CANCELLED']:
            break
    
        time.sleep(1)
    
    if state != 'SUCCEEDED':
        raise Exception(f"Query failed with state: {state}")
    
    results = athena.get_query_results(QueryExecutionId=query_execution_id)
    return parse_athena_results(results)

In [20]:
import json

search_for_fields = [
    "Bnumber"
  #  "AdminHistory",
    # "CustodialHistory",
   # "Acquisition",
    # "Appraisal",
    # "Accruals",
    # "RelatedMaterial",
    # "UserWrapped4", 
    # "Copyright", 
    # "Arrangement", 
    # "Copies", 
    # "Notes", 
    # "Originals"
]

for w in calm_works:
    data = w['data']
    if all(data.get(f, [''])[0].strip() != '' for f in search_for_fields):
        field_content = data[search_for_fields[0]][0]
        calm_ref_no = data['RefNo'][0]

        results = run_axiell_adapter_query(f"SELECT * FROM axiell_adapter_table WHERE content LIKE '%(Calm RefNo){calm_ref_no}%' LIMIT 10;")
        if len(results) > 0:
            collect_id = results[0]['id']
            print(calm_ref_no, field_content, collect_id)
            print(get_oai_raw(collect_id))
            break
        else:
            print("NOT FOUND", calm_ref_no)

PPAEM/F/100/123 b17728320 collect:110109263
<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" priref="110109263" created="2026-05-16T07:11:11Z" modification="2026-05-16T07:11:12Z" selected="false" deleted="false">
  <accession_status>
    <value lang="neutral">OPEN</value>
  </accession_status>
  <current_location.barcode/>
  <current_location.context>215/215;B11/215;B11;MR/215;B11;MR;89/215;B11;MR;89;2/215;B11;MR;89;2;7</current_location.context>
  <current_location.lref>20355</current_location.lref>
  <current_location.name>215;B11;MR;89;2;7</current_location.name>
  <current_location.package_location>
    <value lang="neutral">LOCATION</value>
    <value lang="0">location</value>
    <value lang="1">standplaats</value>
    <value lang="3">Standort</value>
    <value lang="9">plats</value>
    <value lang="10">מיקום</value>
    <value lang="11">placering</value>
    <value lang="14">位置</value>
  </current_location.package_locat

In [73]:
from collections import defaultdict
all_note_fields = [
    "AdminHistory",
    "CustodialHistory",
    "Acquisition",
    "Appraisal",
    "Accruals",
    "RelatedMaterial",
    "UserWrapped4", 
    "Copyright", 
    "Arrangement", 
    "Copies", 
    "Notes", 
    "Originals"
]

works_by_field_count = defaultdict(list)


def count_field_occurences(field_name):
    counter = 0
    for i, w in enumerate(calm_works):
        data = w['data']
        if data.get(field_name, ['']) != ['']:
            counter += 1

    return counter


def count_note_field_occurences_by_work():
    for i, w in enumerate(calm_works):
        counter = 0
        data = w['data']
        for f in all_note_fields:
            if data.get(f, ['']) != ['']:
                counter += 1

        works_by_field_count[counter].append(data.get('RefNo', [''])[0])


count_note_field_occurences_by_work()

for field in search_for_fields:
    print(field, count_field_occurences(field))

UserWrapped4 111356
Originals 35959


In [76]:
works_by_field_count[10]

['UGC155', 'SABSI', 'PPMCS', 'SASMO', 'UGC188', 'MS1456/7920/7938/1/8']

In [75]:
works_by_field_count.keys()

dict_keys([1, 0, 2, 3, 6, 4, 5, 7, 8, 10, 12, 9])

In [78]:
print(get_oai_raw(110206528))

<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" priref="110206528" created="2026-05-16T12:26:28Z" modification="2026-05-16T13:09:07Z" selected="false" deleted="false">
  <access_category.notes>This collection has been partially catalogued and the catalogued part is available to library members. Some items have access restrictions which are explained in the item-level catalogue records. Due to ongoing work to improve our collections management systems we are unable to facilitate access to uncatalogued material until 2027.</access_category.notes>
  <accession_status>
    <value lang="neutral">RESTRICTIONSAPPLY</value>
  </accession_status>
  <accruals>&lt;p&gt;The following is an interim description of material that has been acquired since this collection was catalogued. This description may change when cataloguing takes place in future:&lt;/p&gt;

&lt;p&gt;1 box of additional papers was received in 2004 (acc. 1240), consisting o